# Fine-tune Laya for ontology alignment and entity concept mapping

This notebook fine-tunes **convaiinnovations/laya**, an open Jev-style decision model, using Laya's actual RLCD objective (policy-gradient term plus soft cross-entropy). It does not train TypeSafe Jev weights. A `soft-ce` ablation is configurable.

**Data:** built by `make_dataset.py` from real biomedical resources: OAEI 2025 anatomy (mouse–human) and OAEI Bio-ML 2026 (NCIT–DOID train references with their candidate pools; the valid split and the gold-stripped test pools form a separate benchmark in `data/test/bioml2026/benchmark/`), Biomappings curated SSSOM mappings, BioNLP-Corpus NER mentions mapped to BioPortal candidates (silver, soft probabilities; `concept_selection` task), and the BC5CDR, NCBI-disease and BioRED normalisation corpora (MeSH, NCBI Gene, NCBI Taxonomy); BioPortal mappings and lookups are optional with an API key. Three tasks: ontology-pair relation classification, mention–candidate acceptance and list-wise concept selection; every case has a `choice` and a `noul` question. `insufficient` examples are constructed (identifier-only concepts, ambiguous mentions without context), and some pair labels are derived from the ontology hierarchy; see the provenance field of each row and `data/dataset_stats.json`. 

Run on Python 3.10+ with a CUDA GPU; a 16–24 GB GPU is a reasonable starting point with the small micro-batch, but memory requirements depend on the environment. CPU is allowed but slow. Internet is needed for installation and checkpoint download. No paid API or publishing is required.

Build the data first with `./build_all.sh` (downloads are cached under `cache/`); upload the whole folder in Colab. GPU training was not executed when this notebook was prepared; source/API and dataset checks were performed.


In [ ]:
import os, sys, subprocess
os.environ["USE_TF"] = "0"
# Pin the inspected source rather than a moving main branch.
LAYA_COMMIT = "80a1da1fa5448ea12163f4d977935625a0aea670"
subprocess.check_call([sys.executable, "-m", "pip", "install",
    f"laya @ git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}",
    "scikit-learn", "pandas"])
# Restart the kernel if an older laya/transformers was already imported.


## 1. Data and label meanings

`source_narrower` means source ⊂ target; `source_broader` means target ⊂ source. `related` is an explicit association, not a subclass or equivalence assertion. `no_match` means the evidence supports distinct concepts, **not** an OWL disjointness axiom. `insufficient` is a decision to abstain because evidence is missing.

The binary question asks whether the supplied evidence supports acceptance; a negative answer can mean either a mismatch or insufficient evidence. Use the categorical answer to distinguish them. One-hot targets are hard labels encoded in the required distribution format; they are not measured confidence probabilities.

Splits are by `group_id`: for alignment, connected components of the equivalence graph anchored on the more specific concept (a pair and its reverse share a group); for linking, the PubMed ID / sentence id, merged across sources that share a sentence. A concept can still appear in different groups as a parent or negative, and the same concept may occur in both tasks, so this is not an ontology-disjoint benchmark; use `--holdout-sources` (e.g. `oaei_anatomy`) for an unseen-source test.


In [ ]:
# Data: built by ./build_all.sh into data/train/<dataset>/ and data/test/<dataset>/ (see README).
# training/load_data.load_splits() merges the selected datasets and drops cross-dataset leakage.
import json, os, sys
from pathlib import Path
ROOT = Path(os.environ.get("ROOT") or (Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()))
sys.path.insert(0, str(ROOT / "training"))
from load_data import load_splits, available
DATASETS = os.environ.get("DATASETS", "all")      # e.g. "bioml2026,oaei_anatomy,ner_bionlp"
print("available datasets:", available(ROOT))
rows, load_report = load_splits(ROOT, DATASETS, return_report=True)


In [ ]:
from collections import Counter
import pandas as pd

groups = {s: {r["group_id"] for r in rs} for s,rs in rows.items()}
assert all(not groups[a] & groups[b] for a,b in
           [("train","calibration"),("train","test"),("calibration","test")])
all_ids = [r["id"] for rs in rows.values() for r in rs]
assert len(all_ids) == len(set(all_ids))
for rs in rows.values():
    for r in rs:
        assert set(r["questions"]) == set(r["gold"])
        for qid,q in r["questions"].items():
            p = r["gold"][qid]["probabilities"]
            keys = {"false", "true"} if q["type"] == "noul" else set(q["criteria"])
            assert set(p) == keys and abs(sum(p.values()) - 1) < 1e-6
            assert all(0 <= v <= 1 for v in p.values())
            assert r["gold"][qid]["label"] in keys
print(pd.DataFrame([{ "split":s, "cases":len(rs),
    "decisions":sum(len(r["questions"]) for r in rs), "groups":len(groups[s])}
    for s,rs in rows.items()]))
print(json.dumps(rows["train"][0], indent=2))


## 2. Load the original checkpoint and check token budgets

Fine-tune the English base checkpoint; this example does not assume that the business-workflow fine-tune transfers to ontology mapping. `max_len=512` and `head_max_len=192` match the base defaults. The preflight check below rejects truncated states for this small demo. For real definitions, tune budgets and verify that candidate definitions, relations, and context survive tokenization.


In [ ]:
import random, numpy as np, torch
from huggingface_hub import snapshot_download, model_info
from laya.train import (TrainConfig, load_checkpoint, items_from_rows, train_model,
    calibration_records, evaluate_records, save_checkpoint)
from laya.calibrate import fit_temperature_map
from laya.common import serialize_state, encode_text

random.seed(42); np.random.seed(42); torch.manual_seed(42)
MODEL_ID = "convaiinnovations/laya"
# Resolve and record an exact checkpoint revision for this run.
MODEL_REVISION = model_info(MODEL_ID).sha
model_dir = snapshot_download(MODEL_ID, revision=MODEL_REVISION)
model, tok, cfg = load_checkpoint(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN, HEAD_MAX_LEN = 512, 192
model.to(device)
items = {}
for split, rs in rows.items():
    state_lengths = [len(encode_text(tok, serialize_state(r["state"]))["input_ids"]) for r in rs]
    assert max(state_lengths) < MAX_LEN - HEAD_MAX_LEN - 8, "Shorten states or adjust budgets."
    items[split], skipped = items_from_rows(tok, rs, MAX_LEN, HEAD_MAX_LEN)
    assert not any(skipped.values()), (split, skipped)
    assert len(items[split]) == sum(len(r["questions"]) for r in rs)
    print(split, len(items[split]), "decisions; longest state:", max(state_lengths))
print("Device:", device, "Model revision:", MODEL_REVISION)


## 3. Fine-tune using RLCD

The default runs four epochs of full encoder/head fine-tuning on one device. `micro_batch=2`, `grad_accum=8` gives an effective batch up to 16 decisions. Reduce micro-batch to 1 if needed. Set `freeze_encoder=True` for a lighter head-only experiment; it is a different adaptation setting.

`loss="rlcd"` invokes upstream RLCD, including its soft cross-entropy guidance. Set `loss="soft-ce"` for a separately initialized baseline run, using a different output directory. Do not infer calibration guarantees from the name RLCD. We fit temperatures only on the separate calibration set and evaluate on test data afterwards. The upstream convenience `finetune()` splits individual decisions internally, so this notebook uses its lower-level training API to preserve family-level separation.


In [ ]:
OUTPUT_DIR = ROOT / "models" / "LayaLink"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG = OUTPUT_DIR / "training" / "training_log.jsonl"          # same log format as the scripts
LOG.parent.mkdir(parents=True, exist_ok=True)
import time
def log(**rec):
    with open(LOG, "a") as f:
        f.write(json.dumps(dict(rec, time=round(time.time(), 1)), default=float) + "\n")
train_cfg = TrainConfig(epochs=4, micro_batch=2, grad_accum=8,
    encoder_lr=2.5e-5, head_lr=1e-4, loss="rlcd", rl_samples=4,
    sigma_start=0.4, sigma_end=0.1, shuffle_options=("choice",),
    max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN, seed=42, log_every=20,
    amp=device.type == "cuda", gradient_checkpointing=device.type == "cuda")
train_cfg.validate()

log(event="start", epoch=0, update=0, epochs=4, datasets=DATASETS, train_items=len(items["train"]))
def checkpoint_epoch(epoch, loss):
    log(event="epoch_end", epoch=epoch + 1, avg_loss=float(loss))
    save_checkpoint(model, tok, dict(cfg, max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN),
                    str(OUTPUT_DIR / "checkpoint_latest"))

history = train_model(model, tok, items["train"], train_cfg, device,
    MAX_LEN, HEAD_MAX_LEN, on_epoch_end=checkpoint_epoch)
print("Epoch losses:", history)
sys.path.insert(0, str(ROOT / "training"))
from plot_training import plot
print("training curves:", plot(OUTPUT_DIR))        # models/LayaLink/training/training_curves.png


## 4. Calibrate and save

Calibration uses the union of every dataset's calibration split (counts printed by the data cell). It mixes curated, ontology-derived and silver rows, so temperatures and confidence thresholds learned here should not be used as production guarantees. Get enough representative, expert-labeled calibration examples before deploying.


In [ ]:
cal_records = calibration_records(model, tok, items["calibration"], device,
    MAX_LEN, HEAD_MAX_LEN, batch_size=2)
fitted = fit_temperature_map(cal_records)
out_cfg = dict(cfg, max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN,
               fine_tuned=True, model_name="LayaLink")
out_cfg["temperature"] = fitted["temperature"]
out_cfg.pop("temperature_by_options", None)
if fitted.get("temperature_by_options"):
    out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
save_checkpoint(model, tok, out_cfg, str(OUTPUT_DIR))
print("Fitted temperatures:", fitted)


## 5. Held-out evaluation

Compare raw and calibrated fine-tuned predictions on the same test decisions. These are custom splits, not an official benchmark; inspect per-task, per-label and per-dataset results. For the Bio-ML 2026 comparison, run `bash evaluation/run_evaluation.sh` on `data/test/bioml2026/benchmark/` (local MRR/Hits@k on the public valid split, plus a CodaBench submission file for the hidden test). To compare the pretrained baseline, reload the base checkpoint in a separate run and score the same test split. Keep test data out of hyperparameter and threshold selection.


In [ ]:
test_records = calibration_records(model, tok, items["test"], device,
    MAX_LEN, HEAD_MAX_LEN, batch_size=2)
report = {
    "datasets": load_report["datasets"], "dropped_for_leakage": load_report["dropped"], "laya_commit": LAYA_COMMIT,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "training_loss": history,
    "raw": evaluate_records(test_records),
    "calibrated": evaluate_records(test_records, out_cfg["temperature"],
                                  out_cfg.get("temperature_by_options")),
}
(OUTPUT_DIR / "evaluation.json").write_text(json.dumps(report, indent=2, default=float))
print(json.dumps(report, indent=2, default=float))


## 6. Reload, predict, and inspect each task

A probability is a model estimate, not proof of logical equivalence. This model judges supplied candidate pairs; it does not search the ontology or verify global consistency. Retrieve candidates first, score pairs, and retain the original evidence and ontology versions. Check logical consequences separately before asserting `owl:equivalentClass`.


In [ ]:
del model
a = None
if torch.cuda.is_available(): torch.cuda.empty_cache()
import laya
from sklearn.metrics import classification_report
agent = laya.Agent(str(OUTPUT_DIR), device=str(device))
predictions = []
for r in rows["test"]:
    answers = agent.predict(r["state"], r["questions"])["answers"]
    predictions.append({"id":r["id"], "task":r["task"], "answers":answers, "gold":r["gold"]})
for task, qid in [("ontology_alignment", "relation"), ("entity_linking", "mapping"), ("concept_selection", "concept")]:
    subset=[r for r in predictions if r["task"]==task]
    if not subset: continue
    print(task)
    print(classification_report([r["gold"][qid]["label"] if task != "concept_selection" else ("none" if r["gold"][qid]["label"] == "none" else "candidate") for r in subset],
          [r["answers"][qid]["choice"] if task != "concept_selection" else ("none" if r["answers"][qid]["choice"] == "none" else "candidate") for r in subset], zero_division=0))
(OUTPUT_DIR / "predictions.json").write_text(json.dumps(predictions, indent=2))

example = next(r for r in rows["test"] if r["task"] == "ontology_alignment")
print(json.dumps(example["state"], indent=2))
print(json.dumps(agent.predict(example["state"], example["questions"])["answers"], indent=2))


## 7. Replace the demonstration data

Keep the JSONL `state`, `questions`, `gold` structure. Add ontology identifiers, versions, definitions, synonyms, parent classes and relevant restrictions to your real input, while preserving useful information within the token budget. Keep annotation rationale and provenance as metadata outside `state` unless intentionally provided at inference.

- Collect positive equivalences plus hard negatives: siblings, broader/narrower concepts, part-of relations, homonyms and species-specific mismatches.
- Add no-candidate-found cases upstream; a pairwise judge cannot recover a missing candidate. Measure retrieval recall@K separately from mapping precision/recall.
- For entity linking, use real mention spans and surrounding sentences. Avoid systematic label or candidate-position clues.
- Annotate uncertainty explicitly. Do not manufacture soft probabilities such as 0.95 from a hard annotation. Multiple qualified annotators can supply empirical vote distributions, with disagreement documented.
- Use separate train, calibration, validation (if tuning), and final test sets. Group duplicate concepts, reverse pairs and text variants before splitting. This small starter has no separate tuning set.
- Monitor `noul` for constant or overconfident predictions. The model card documents a false/true label sensitivity issue. Compare a binary `choice` formulation with neutral option keys if needed, and retrain/evaluate that schema consistently.
- Choose auto-accept thresholds on representative held-out calibration/validation data and assess precision versus coverage. Do not hard-code 0.9 as a safe threshold.

Sources inspected for implementation:
- [Model card](https://huggingface.co/convaiinnovations/laya)
- [Pinned training implementation](https://github.com/NandhaKishorM/laya/blob/80a1da1fa5448ea12163f4d977935625a0aea670/laya/train.py)
- [Fine-tuning documentation](https://github.com/NandhaKishorM/laya/blob/80a1da1fa5448ea12163f4d977935625a0aea670/docs/finetune.md)

No model weights are included. All sample rows were generated for this tutorial and are freely reusable; the upstream model/code retain their own licenses.
